### Soluția pentru XOR? Adăugăm adâncime (Deep Learning)

Am văzut că un singur neuron nu poate rezolva problema XOR pentru că nu poate trasa o graniță de decizie curbă. Soluția naturală în Machine Learning este să creăm o rețea mai complexă, punând mai multe straturi de neuroni unul după altul. Acest tip de arhitectură se numește **Multi-Layer Perceptron (MLP)**.

Definim termenii:

* **Strat de Intrare (Input Layer):** Datele noastre brute ($X$).
* **Strat Ascuns (Hidden Layer):** Un strat intermediar de neuroni care procesează intrările.
* **Strat de Ieșire (Output Layer):** Ultimul neuron care dă decizia finală.

### There is a catch!

Marea tentație este să credem că dacă punem un strat ascuns cu 100 de neuroni între intrare și ieșire, rețeaua devine automat de 100 de ori mai deșteaptă.

**Să testăm această ipoteză matematic.** Ce se întâmplă dacă construim un MLP, dar **NU folosim nicio funcție de activare non-liniară** (cum e Sigmoid sau ReLU) între straturi? Adică lăsăm neuronii să fie pur liniari ($z = A \cdot W$).

Să presupunem o rețea simplă cu un strat ascuns:

1. **Intrare:** Matricea $X$.
2. **Stratul Ascuns:** Are ponderile $W_1$. Calculează rezultatul intermediar $H$:

$$H = X \cdot W_1$$


3. **Stratul de Ieșire:** Ia rezultatul $H$ și îl înmulțește cu propriile ponderi $W_2$ pentru a obține rezultatul final $Y$:

$$Y = H \cdot W_2$$



### Demonstrația Matematică a Colapsării

Dacă combinăm cele două ecuații de mai sus, înlocuindu-l pe $H$ în a doua formulă, obținem:

$$Y = (X \cdot W_1) \cdot W_2$$

În algebra liniară, înmulțirea matricelor este *asociativă*. Asta înseamnă că putem muta parantezele:

$$Y = X \cdot (W_1 \cdot W_2)$$

Acum, privește paranteza $(W_1 \cdot W_2)$. Rezultatul înmulțirii a două matrice de ponderi este... pur și simplu o altă matrice de ponderi. Să o numim $W_{final}$.

$$W_{final} = W_1 \cdot W_2$$

Ecuația rețelei noastre "complexe" devine:

$$Y = X \cdot W_{final}$$

**Concluzia:** Oricât de multe straturi liniare am pune unul după altul, ele se prăbușesc (colapsează) matematic într-un **singur strat liniar echivalent**. Rețeaua nu a învățat nimic complex, ci doar a făcut o înmulțire matricială mai lungă pentru a trasa tot o linie dreaptă. De aceea avem nevoie de activări non-liniare între straturi.

---

### Exercițiu: Demonstrarea Colapsării în Cod


În acest exercițiu, vom crea o rețea cu un strat ascuns uriaș (50 de neuroni), dar fără activări. Vom calcula ieșirea trecând prin ambele straturi, apoi vom calcula ieșirea folosind o singură matrice "colapsată" și vom demonstra că rezultatele sunt identice.

In [ ]:
import numpy as np

# Setăm o sămânță aleatoare pentru rezultate reproductibile
np.random.seed(101)

# A: Date de intrare (2 exemple, 2 caracteristici)
A_intrare = np.array([[0.5, 0.2],
                     [0.1, 0.9]])

# ARHITECTURA REȚELEI LINIARĂ:
# Intrare (2) -> Strat Ascuns (50 neuroni) -> Ieșire (1)

# W1: Ponderi Strat Ascuns (2 intrări x 50 neuroni)
W1 = np.random.randn(2, 50)
b1 = np.random.randn(50) # Bias-uri strat ascuns

# W2: Ponderi Strat Ieșire (50 intrări de la stratul ascuns x 1 neuron ieșire)
W2 = np.random.randn(50, 1)
b2 = np.random.randn(1) # Bias strat ieșire


# ---  Calculăm trecând prin ambele straturi ---

# Z1: Rezultat Strat Ascuns (Liniar, fără activare!)
Z1 = ...

# Y_lung: Rezultat Final la ieșire (Liniar, fără activare!)
Y_lung = ...


# --- PASUL 2: Calculăm folosind matricea colapsată (Shortcut Matematic) ---

# Calculăm ponderile echivalente: W_final = W1 @ W2
W_final = ...

# Calculăm bias-ul echivalent (matematica e puțin mai complexă aici din cauza b1)
b_final = (b1 @ W2) + b2

# Y_scurt: Rezultat Final direct
Y_scurt = ...


# --- VERIFICARE ---

print("Rezultate obținute prin pipeline-ul lung (2 straturi):")
print(Y_lung)

print("\nRezultate obținute prin shortcut-ul colapsat (1 strat echivalent):")
print(Y_scurt)

# Verificăm dacă sunt identice (folosim np.allclose din cauza micilor erori de precizie floating point)
sunt_identice = np.allclose(Y_lung, Y_scurt)

print(f"\nSunt rezultatele identice? {sunt_identice}")

### Primi pasi catre deep learning - Magia Non-Liniarității

Am văzut dezastrul: fără o funcție specială între straturi, 100 de straturi se comportă exact ca unul singur. Rețeaua noastră este rigidă și blocată într-o lume a liniilor drepte.

Pentru a o "salva" și a-i permite să învețe forme curbe, granițe neregulate și soluția pentru XOR, trebuie să aplicăm o **funcție de activare non-liniară** după prima înmulțire matricială.

În Deep Learning-ul modern, cea mai populară funcție pentru straturile ascunse (Hidden Layers) nu este Sigmoid, ci **ReLU (Rectified Linear Unit)**.

**Intuiția din spatele ReLU:**
Gândește-te la ReLU ca la un paznic foarte strict la un club. Dacă semnalul (numărul) care vine de la neuron este negativ (mai mic ca 0), paznicul îl blochează complet și îl transformă în 0. Dacă semnalul este pozitiv, paznicul îl lasă să treacă exact așa cum este.

Formula matematică este incredibil de simplă:


$$f(z) = \max(0, z)$$

De ce este atât de populară? Pentru că este foarte rapid de calculat pentru calculator, iar gradientul ei (panta) nu dispare niciodată pentru numerele pozitive, ajutând rețelele adânci să învețe mult mai repede.

---

### Exercițiu: Implementarea și Vizualizarea ReLU



In [ ]:

import numpy as np
import matplotlib.pyplot as plt

def relu(z):
    # Folosim np.maximum pentru a alege maximul dintre 0 și z
    return np.maximum(0, z)

z_valori = np.linspace(-10, 10, 100)
y_relu = relu(z_valori)

plt.figure(figsize=(8, 5))
plt.plot(z_valori, y_relu, linewidth=3, color='orange')
plt.title("Funcția ReLU (Rectified Linear Unit)")
plt.xlabel("Intrare (z)")
plt.ylabel("Ieșire (y)")
plt.grid(True)
plt.show()


### Flavours of ReLU: Rezolvarea Neuronului Mort (Dying ReLU)


Deși ReLU este vedeta industriei, are o vulnerabilitate.
Dacă ponderile se actualizează într-un mod nefericit, un neuron ar putea ajunge să producă doar numere negative pentru toate datele din setul tău. Când trec prin ReLU, toate aceste numere devin $0$. Mai mult, derivata (gradientul) pentru zero este $0$.

Neuronul devine orb și surd. Nu mai transmite nimic și nu mai învață nimic. Aceasta este problema **Neuronului Mort (Dying ReLU)**.

Pentru a repara asta, cercetătorii au creat variante ("arome") ale funcției:

**1. Leaky ReLU (ReLU cu scurgere)**
În loc să blocheze complet valorile negative la zero, Leaky ReLU le permite să aibă o pantă foarte mică (o mică "scurgere", de obicei înmulțind numărul cu $0.01$).


$$f(z) = \begin{cases}  z & \text{dacă } z > 0 \\ 0.01 \cdot z & \text{dacă } z \leq 0  \end{cases}$$


Acum, chiar dacă neuronul cade în zona negativă, gradientul nu mai este zero absolut, deci are o șansă "să se trezească".

**2. ELU (Exponential Linear Unit)**
O variantă și mai fină, care curbează ușor valorile negative folosind funcția exponențială, făcând tranziția mult mai naturală și netedă sub axa 0.

---

### Alte Funcții Clasice: Tanh (Tangenta Hiperbolică)

În afară de familia ReLU, o altă funcție istorică foarte importantă este **Tanh**.

Ea este "vărul mai deștept" al lui Sigmoid. Dacă Sigmoid "strivește" numerele într-un interval între $0$ și $1$, Tanh le strivește între **$-1$ și $1$**.

Formula:


$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

**De ce este mai bună ca Sigmoid pentru straturile ascunse?**
Deoarece este centrată în jurul lui $0$. Dacă o rețea primește valori centrate în jurul lui zero, calculele matematice din timpul antrenării devin mult mai stabile, iar Gradient Descent coboară muntele mai rapid și mai uniform.

---

### Exercițiu: Comparație Vizuală (Leaky ReLU vs. Tanh)

In [ ]:

# Definim funcțiile
def leaky_relu(z, alpha=0.1): # Folosim alpha=0.1 pentru a face linia mai vizibilă pe grafic
    return np.where(z > 0, z, alpha * z)

def tanh(z):
    return np.tanh(z)

y_leaky = leaky_relu(z_valori)
y_tanh = tanh(z_valori)

# Creăm două grafice side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graficul 1: Leaky ReLU
ax1.plot(z_valori, y_leaky, linewidth=3, color='magenta')
ax1.set_title("Leaky ReLU")
ax1.axhline(0, color='black', linewidth=1)
ax1.axvline(0, color='black', linewidth=1)
ax1.grid(True)

# Graficul 2: Tanh
ax2.plot(z_valori, y_tanh, linewidth=3, color='purple')
ax2.set_title("Tanh (Tangenta Hiperbolică)")
ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(0, color='black', linewidth=1)
# Adăugăm linii pentru asimptotele -1 și 1
ax2.axhline(1, color='gray', linestyle='--')
ax2.axhline(-1, color='gray', linestyle='--')
ax2.grid(True)

plt.show()

### Bun venit în liga mare: Introducere în PyTorch

Până acum, am scris totul "de mână" folosind NumPy. Am înmulțit noi matricele, am scris noi funcțiile de activare și am calculat matematic derivatele (gradienții) pentru a face modelul să învețe. Asta a fost esențial ca să înțelegi exact ce se întâmplă sub capotă.

Dar în lumea reală, nu scrie nimeni gradienții de mână! Ar fi un coșmar matematic pentru o rețea cu milioane de parametri.
Aici intervine **PyTorch**.

PyTorch este un framework de Deep Learning care face două lucruri magice:

1. **Tensiunea (Tensors):** Funcționează exact ca `np.array`, dar Tensiunile (Tensors) din PyTorch pot rula direct pe plăcile video (GPU) pentru a face calcule de mii de ori mai rapid.
2. **Autograd (Automatic Differentiation):** PyTorch reține absolut orice operație matematică faci. Când îi spui `.backward()`, calculează el **automat** toți gradienții pentru toată rețeaua. Fără Chain Rule scris pe hârtie!

---

### Cum arată un Perceptron în PyTorch?

Pentru a construi un model în PyTorch, folosim mereu o rețetă standard: creăm o clasă care moștenește din `torch.nn.Module`.

Această clasă are mereu două părți obligatorii:

* `__init__`: Aici declarăm **piesele de Lego**. Ce straturi vrem să folosim? (Aici PyTorch creează automat ponderile $W$ și bias-ul $b$ și le inițializează cu valori aleatoare).
* `forward`: Aici spunem **cum se leagă piesele**. În ce ordine trece informația $X$ prin ele?

Nu mai scriem $X \cdot W + b$. În PyTorch, această combinație liniară se numește simplu: **`nn.Linear`**.

Haideți să recreăm exact primul nostru model (Perceptronul cu un singur strat) pentru problema XOR.

---

### Exercițiu: Primul tău model în PyTorch (Single Layer)



In [1]:

import torch
import torch.nn as nn

# 1. Datele noastre (folosim torch.tensor în loc de np.array)
# În PyTorch, e important să specificăm că lucrăm cu numere cu virgulă (float32)
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]], dtype=torch.float32)

Y = torch.tensor([[0.0],
                  [1.0],
                  [1.0],
                  [0.0]], dtype=torch.float32)

# 2. Definim Rețeaua Neuronală
class SingleLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear este exact operația: Z = X @ W + b
        # in_features = 2 (avem 2 coordonate pentru fiecare punct)
        # out_features = 1 (vrem un singur neuron final)
        self.strat_liniar = nn.Linear(in_features=2, out_features=1)

    def forward(self, x):
        # Aici definim traseul datelor
        z = self.strat_liniar(x)
        return z

# 3. "Dăm viață" modelului
model_simplu = SingleLayerPerceptron()

# 4. Trecem datele prin model (Forward Pass)
# În PyTorch nu mai trebuie să apelăm model.forward(X), pur și simplu scriem model(X)
rezultat_brut = model_simplu(X)

print("Rezultatele brute (Z) prezise de modelul PyTorch neantrenat:")
print(rezultat_brut)

# Opțional: Dacă vrem să vedem ce ponderi a ales PyTorch aleatoriu:
print("\nPonderile (W) și Bias-ul (b) generate automat:")
for nume, parametru in model_simplu.named_parameters():
    print(f"{nume}: {parametru.data}")

Rezultatele brute (Z) prezise de modelul PyTorch neantrenat:
tensor([[0.0119],
        [0.5737],
        [0.3762],
        [0.9380]], grad_fn=<AddmmBackward0>)

Ponderile (W) și Bias-ul (b) generate automat:
strat_liniar.weight: tensor([[0.3643, 0.5618]])
strat_liniar.bias: tensor([0.0119])



### Multi-Layer Perceptron (MLP) în PyTorch

Este timpul să construim rețeaua care va învinge problema XOR. Așa cum am discutat teoretic, avem nevoie de un **Strat Ascuns (Hidden Layer)** și de o **activare non-liniară (ReLU)** între straturi.

**Arhitectura noastră va arăta așa:**

1. **Intrare (2 neuroni):** Primește coordonatele $(x_1, x_2)$.
2. **Strat Ascuns (4 neuroni):** Primul strat liniar. Am ales 4 neuroni (poti alege si 2 sau 8), e suficient pentru a "curba" spațiul XOR.
3. **Activare ReLU:** Pasul magic care introduce non-liniaritatea după stratul ascuns.
4. **Strat de Ieșire (1 neuron):** Al doilea strat liniar care strânge informațiile.
5. **Activare Sigmoid:** Deoarece avem o clasificare binară (0 sau 1), vrem ca ieșirea să fie o probabilitate la final.

---

### Exercițiu: Definirea Arhitecturii MLP


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim # Importăm pachetul pentru optimizatori
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# 1. Datele XOR (aceleași ca înainte)
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]], dtype=torch.float32)

Y = torch.tensor([[0.0],
                  [1.0],
                  [1.0],
                  [0.0]], dtype=torch.float32)

# 2. Definim Rețeaua MLP
class XORModelMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Definim piesele de Lego:

        # Stratul 1
        self.strat_ascuns = ...

        # Activarea ReLU (non-liniară)
        self.relu = ...

        # Stratul 2: Ascuns (4) -> Ieșire (1)
        self.strat_iesire = ...

        # Activarea finală Sigmoid (pentru probabilitate 0-1)
        self.sigmoid = ...

    def forward(self, x):
        # Definim traseul (Forward Pass):
        # X -> Liniar1 -> ReLU -> Liniar2 -> Sigmoid -> Output

        z1 = ...
        a1 = ...
        z2 = ...
        probabilitate = ...

        return probabilitate

# Inițializăm modelul
model_mlp = XORModelMLP()
print("Arhitectura modelului MLP:")
print(model_mlp)

### Ingredientele Antrenării: Criterion și Optimizator

Avem creierul (modelul), dar acum avem nevoie de motorul care îl face să învețe. În PyTorch, acesta este format din două componente esențiale:

**1. Criterion (Funcția de Loss / Eroare)**
Trebuie să dăm o notă modelului. Deoarece ultima noastră activare este Sigmoid și avem de-a face cu o clasificare binară (0 sau 1), cel mai indicat criteriu este **Binary Cross Entropy Loss (BCELoss)**.

* În PyTorch: `nn.BCELoss()`

**2. Optimizatorul (Algoritmul de învățare)**
Optimizatorul este cel care ia gradienții (pantele) calculați automat de PyTorch și actualizează ponderile ($W, b$) pentru a scădea eroarea.
Am cerut **SGD (Stochastic Gradient Descent)**. Deși datele noastre sunt foarte mici (un singur batch de 4 puncte) și tehnic facem *Batch Gradient Descent*, în PyTorch folosim implementarea `optim.SGD` pe tot setul de date.

* În PyTorch: `optim.SGD(model.parameters(), lr=learning_rate)`

### Setup-ul pentru Criterion, Optimizator și Vizualizare

In [3]:

# Setăm o Rată de Învățare (Learning Rate) destul de mare ca să vedem rapid rezultatele
learning_rate = 0.3

# 1. Definim Criteriul (Loss): Binary Cross Entropy
criterion = nn.BCELoss()

# 2. Definim Optimizatorul: SGD, și îi spunem ce parametrii să actualizeze
optimizer = optim.SGD(model_mlp.parameters(), lr=learning_rate)


# Creăm o grilă de puncte de la -0.5 la 1.5 pentru a colora fundalul
pas = 0.02
x_min, x_max = -0.5, 1.5
y_min, y_max = -0.5, 1.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, pas),
                     np.arange(y_min, y_max, pas))

# Transformăm grila într-un Tensor PyTorch pentru a o trece prin model
grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

### The Training Loop

Acum punem totul cap la cap într-o buclă `for` (Epoci). Rețeta standard PyTorch pentru fiecare epocă este următoarea:

1. **`optimizer.zero_grad()`**: Esențial! PyTorch acumulează gradienții implicit. Trebuie să ștergem gradienții vechi de la epoca anterioară înainte de a calcula unii noi.
2. **Forward Pass**: Trecem datele prin model (`P = model(X)`).
3. **Calcul Loss**: Comparăm predicțiile cu realitatea folosind criteriul definit (`loss = criterion(P, Y)`).
4. **Backward Pass (`loss.backward()`)**: Magia Autograd! PyTorch calculează automat toți gradienții pentru fiecare pondere din rețea folosind Chain Rule.
5. **Optimizer Step (`optimizer.step()`)**: Optimizatorul ia gradienții proaspeți și actualizează ponderile: $W = W - lr \times grad$.

---

### Exercițiu Final: Rularea Antrenării

In [ ]:


num_epochs = 2001 # Numărul de treceri prin date

print("Începe antrenamentul MLP pe XOR cu vizualizare...")
time.sleep(2)

for epoch in range(num_epochs):
    # 1. Resetăm gradienții
    optimizer.zero_grad()

    # 2. Forward Pass
    predicții = ...

    # 3. Calculăm Loss-ul
    loss = ...

    # 4. Backward Pass (calculăm gradienții automați)
    ...

    # 5. Actualizăm ponderile (pasul optimizatorului)
    ...

    if epoch % 50 == 0:
        # Trecem modelul în mod evaluare
        ...

        with torch.no_grad(): # Nu calculăm gradienți pentru vizualizare
            # Prezicem pentru toată grila de pe ecran
            Z_grid = model_mlp(grid_tensor)
            Z_grid = Z_grid.reshape(xx.shape).numpy()

        # Revenim la modul antrenare
        ...

        clear_output(wait=True)
        plt.figure(figsize=(10, 8))

        # Desenăm fundalul colorat (Contourf) - arată granița curbă!
        plt.contourf(xx, yy, Z_grid, levels=50, cmap='RdBu', alpha=0.7)
        # Adăugăm linia albă de decizie (unde probabilitatea e fix 0.5)
        plt.contour(xx, yy, Z_grid, levels=[0.5], colors='white', linewidths=3)

        # Desenăm punctele XOR originale
        X_np = X.numpy()
        Y_np = Y.numpy()
        plt.scatter(X_np[Y_np[:,0]==0, 0], X_np[Y_np[:,0]==0, 1], color='red', edgecolor='black', s=200, label='Clasa 0')
        plt.scatter(X_np[Y_np[:,0]==1, 0], X_np[Y_np[:,0]==1, 1], color='blue', edgecolor='black', s=200, label='Clasa 1')

        plt.title(f"Antrenare MLP pe XOR - Epoca {epoch}\nLoss: {loss.item():.6f}\nDatorită ReLU, linia albă s-a curbat și separă punctele!")
        plt.xlim(-0.5, 1.5)
        plt.ylim(-0.5, 1.5)
        plt.legend(loc='lower right')
        plt.grid(True, linestyle='--', alpha=0.5)

        plt.show()

        # O mică pauză ca să fie animația fluidă
        if epoch < 500:
            time.sleep(0.1)
        else:
            time.sleep(0.01)

print("\nAntrenament finalizat cu succes! Modelul a învățat XOR.")